# Learn LangGraph without using any AI models

Focus on just the LangGraph framework


In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from IPython.display import Image, display
import gradio as gr
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from pydantic import BaseModel
import random

## 5 LG steps
LG uses 5 steps to setup and run the framework
1. Define the state Object
2. Start Graph Builder
3. Create a node (function)
4. Create edges (decides which node to call)
5. Compile the Graph

In [ ]:
# 1. define state object using pydantic

class State(BaseModel):
    messages: Annotated[list, add_messages]

In [ ]:
# 2. create graph_builder

graph_builder = StateGraph(State)

In [ ]:
# 3. Create node.  A python function to generate funny random text and its added to graph_builder

nouns = ["Cabbages", "Unicorns", "Toasters", "Penguins", "Bananas", "Zombies", "Rainbows", "Eels", "Pickles", "Muffins"]
adjectives = ["outrageous", "smelly", "pedantic", "existential", "moody", "sparkly", "untrustworthy", "sarcastic", "squishy", "haunted"]

def random_text_generator_node(old_state: State) -> State:
    reply = f"{random.choice(nouns)} are {random.choice(adjectives)}"
    messages = [{"role": "assistant", "content": reply}]
    new_state = State(messages=messages)
    return new_state

graph_builder.add_node("random_text_generator_node", random_text_generator_node)

In [ ]:
# 4. add edgens.  START and END are defined in LG library

graph_builder.add_edge(START, "random_text_generator_node")
graph_builder.add_edge("random_text_generator_node", END)


In [ ]:
# 5. Compile graph and display it as an img

graph = graph_builder.compile()

display(Image(graph.get_graph().draw_mermaid_png()))


## Put it all together in Gradio UI

In [ ]:
def chat(user_input: str, history):
    message = {"role": "user", "content": user_input}
    messages = [message]
    state = State(messages=messages)
    result = graph.invoke(state)
    print(result)
    return result["messages"][-1].content


gr.ChatInterface(chat).launch()